# Predicting term-deposit subscriptions

**Question:** Can information available before a marketing call ends predict whether a client subscribes (`y`)?

`duration` is deliberately excluded from the deployable model because it is only known after the call. Using it would leak future information into a model intended to prioritize clients before or during contact. A separate, explicitly leaky model quantifies how much this can inflate performance.

The positive class is uncommon, so this notebook emphasizes ROC-AUC and average precision (PR-AUC), not raw accuracy. The decision threshold is selected on validation data and evaluated once on the held-out test set.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

In [ ]:
data_path = Path("data.csv")
df = pd.read_csv(data_path)

assert "y" in df.columns and "duration" in df.columns
assert df["y"].isin(["yes", "no"]).all()

print(f"Rows: {len(df):,}; columns: {df.shape[1]}")
display(df.head())
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(1),
    "unique": df.nunique(dropna=True),
}))
print("Target distribution:")
display(df["y"].value_counts().rename("count").to_frame().assign(
    percent=lambda x: (100 * x["count"] / x["count"].sum()).round(2)
))

## Leakage-safe experimental design

The rows are split 60%/20%/20% into train, validation, and test sets with the class ratio preserved. Preprocessing is fitted inside each pipeline, so imputation, scaling, and one-hot category discovery use training data only.

This dataset has no client identifier or timestamp with a year, so we cannot test for repeated clients or perform a true future-time split. The random split therefore estimates performance on calls drawn from a similar population, not necessarily a future campaign.

In [ ]:
y = df["y"].eq("yes").astype(int)
X = df.drop(columns="y")

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(y_train), len(y_valid), len(y_test)],
    "positive_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
}, index=["train", "validation", "test"])
display(split_summary.style.format({"positive_rate": "{:.2%}"}))

In [ ]:
def make_pipeline(columns, estimator):
    numeric = X_train[columns].select_dtypes(include="number").columns.tolist()
    categorical = [c for c in columns if c not in numeric]
    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])
    return Pipeline([("preprocess", preprocess), ("model", estimator)])

safe_columns = [c for c in X.columns if c != "duration"]
all_columns = X.columns.tolist()

models = {
    "dummy_prevalence": make_pipeline(
        safe_columns, DummyClassifier(strategy="prior")
    ),
    "logistic_safe": make_pipeline(
        safe_columns, LogisticRegression(max_iter=2000, solver="liblinear")
    ),
    "logistic_with_duration_LEAKY": make_pipeline(
        all_columns, LogisticRegression(max_iter=2000, solver="liblinear")
    ),
}

for name, model in models.items():
    columns = all_columns if "LEAKY" in name else safe_columns
    model.fit(X_train[columns], y_train)
    print(f"Fitted {name}")

## Validation comparison

The leaky model is a diagnostic only. It must not be selected for deployment, regardless of its score. Average precision should be compared with the positive-class prevalence (the dummy baseline).

In [ ]:
def probabilities(model_name, X_part):
    columns = all_columns if "LEAKY" in model_name else safe_columns
    return models[model_name].predict_proba(X_part[columns])[:, 1]

validation_scores = {}
rows = []
for name in models:
    score = probabilities(name, X_valid)
    validation_scores[name] = score
    rows.append({
        "model": name,
        "ROC-AUC": roc_auc_score(y_valid, score),
        "average_precision": average_precision_score(y_valid, score),
    })
validation_results = pd.DataFrame(rows).set_index("model").sort_values(
    "average_precision", ascending=False
)
display(validation_results.style.format("{:.3f}"))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, score in validation_scores.items():
    RocCurveDisplay.from_predictions(y_valid, score, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_valid, score, name=name, ax=axes[1])
axes[0].set_title("Validation ROC curves")
axes[1].set_title("Validation precision-recall curves")
plt.tight_layout()
plt.show()

## Choose an operating threshold without touching the test set

A 0.5 cutoff is not inherently correct. For illustration, we choose the safe model's threshold that maximizes F1 on validation data. In production, this should instead reflect the relative value of a subscription and cost of contacting a client, or a fixed call-center capacity.

In [ ]:
safe_valid_score = validation_scores["logistic_safe"]
precision, recall, thresholds = precision_recall_curve(y_valid, safe_valid_score)
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
best_idx = np.nanargmax(f1)
chosen_threshold = thresholds[best_idx]
print(f"Validation-selected threshold: {chosen_threshold:.3f}")
print(f"Validation F1 at threshold: {f1[best_idx]:.3f}")

## Final held-out evaluation

Only the leakage-safe model is evaluated operationally below. The test set was not used for fitting or threshold selection.

In [ ]:
safe_test_score = probabilities("logistic_safe", X_test)
safe_test_pred = (safe_test_score >= chosen_threshold).astype(int)

test_metrics = pd.Series({
    "ROC-AUC": roc_auc_score(y_test, safe_test_score),
    "average_precision": average_precision_score(y_test, safe_test_score),
    "F1_at_validation_threshold": f1_score(y_test, safe_test_pred),
    "threshold": chosen_threshold,
}, name="safe model test result")
display(test_metrics.to_frame().style.format("{:.3f}"))
print(classification_report(y_test, safe_test_pred, target_names=["no", "yes"], digits=3))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, safe_test_pred), display_labels=["no", "yes"]
).plot(cmap="Blues")
plt.title("Leakage-safe model: held-out test set")
plt.show()

## Interpret the safe model

Logistic-regression coefficients describe associations after preprocessing; they are not causal effects. Positive values push predictions toward subscription, while negative values push them away.

In [ ]:
safe_model = models["logistic_safe"]
feature_names = safe_model.named_steps["preprocess"].get_feature_names_out()
coefficients = pd.Series(
    safe_model.named_steps["model"].coef_[0], index=feature_names, name="coefficient"
).sort_values()

important = pd.concat([coefficients.head(10), coefficients.tail(10)])
colors = np.where(important < 0, "#C44E52", "#4C72B0")
important.plot.barh(figsize=(9, 7), color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Largest safe-model coefficients")
plt.xlabel("Log-odds coefficient")
plt.tight_layout()
plt.show()

In [ ]:
safe_ap = validation_results.loc["logistic_safe", "average_precision"]
leaky_ap = validation_results.loc["logistic_with_duration_LEAKY", "average_precision"]
safe_auc = test_metrics["ROC-AUC"]
safe_test_ap = test_metrics["average_precision"]

print("CONCLUSION")
print("----------")
print(
    f"Yes: the leakage-safe model ranks subscribers better than chance on held-out data "
    f"(test ROC-AUC={safe_auc:.3f}, average precision={safe_test_ap:.3f}, "
    f"versus prevalence={y_test.mean():.3f})."
)
print(
    f"Including duration raises validation average precision from {safe_ap:.3f} to "
    f"{leaky_ap:.3f}, but that result is not deployable before the call ends."
)
print(
    "Before production: validate on a genuinely later campaign, check for repeated clients, "
    "choose the threshold from business costs/capacity, assess calibration and subgroup performance, "
    "and monitor drift."
)